#01. Persona Vector Extraction (2-Axis)

목적: 경제축(Economic Left)과 사회축(Social Progressive) 페르소나 벡터 추출
모델: Qwen2.5-7B-Instruct (주 모델). 다른 모델은 Cell 1의 MODEL_CONFIGS 참고.
출력: persona_vectors/{model_name}/{trait}_response_avg_diff.pt 등 3종 + best layer info

실행 순서:

    Cell 1: 설정 (MODEL_KEY, AXIS 지정)
    Cell 2~끝: 순서대로 실행
    economic 완료 후 AXIS = "social" 바꿔서 다시 Cell 1부터 재실행


*주의: 각각 instruction(pos/neg 5쌍), questions(40문항), eval prompt 로드해야 함, 즉 폴더에 저장되어 있어야 함. 01 Cell 3의 prepare_axis_data()가 이걸 읽어서 economic_left.json / social_progressive.json을 extract 20 + eval 20으로 쪼개 만들기 때문
data_generation/trait_data_extract/data_economic.json
data_generation/trait_data_extract/data_social.json


In [ ]:
# ── Cell 1: 설정 ────────────────────────────────────────────────
import os
from pathlib import Path
# --- repo-relative project root: works on any machine ---
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)

MODEL_KEY = "qwen"      # "llama" / "qwen"
AXIS      = "social"   # "economic" / "social"

VECTOR_TYPE    = "response_avg"   # 연구 설계상 사전 고정
N_PER_Q        = 10             # 질문당 rollout (do_sample=True + seed)
BATCH_SIZE     = 32
MAX_NEW_TOKENS = 512
JUDGE_MODEL    = "gpt-4.1-mini-2025-04-14"
MAX_CONCURRENT_JUDGES = 50
THRESHOLD      = 50

SKIP_GENERATION = False
SKIP_VECTOR     = False

DATA_JSON = {
    "economic": "data_generation/trait_data_extract/data_economic.json",
    "social": "data_generation/trait_data_extract/data_social.json",
}[AXIS]

MODEL_CONFIGS = {
    "llama": {"model_id": "meta-llama/Llama-3.1-8B-Instruct",
              "short_name": "llama-3.1-8b-instruct", "trust_remote_code": False},
    "qwen":  {"model_id": "Qwen/Qwen2.5-7B-Instruct",
              "short_name": "qwen2.5-7b-instruct",  "trust_remote_code": True},
}
AXIS_CONFIGS = {
    "economic": {"trait_name": "economic_left",        "trait_label": "Economic Left"},
    "social": {"trait_name": "social_progressive", "trait_label": "Social Progressive"},
}

cfg        = MODEL_CONFIGS[MODEL_KEY]
axis_cfg   = AXIS_CONFIGS[AXIS]
TRAIT_NAME  = axis_cfg["trait_name"]
TRAIT_LABEL = axis_cfg["trait_label"]

print(f"모델: {cfg['model_id']}")
print(f"축  : {AXIS} ({TRAIT_LABEL})")
print(f"벡터: {VECTOR_TYPE}")

In [ ]:
# ── Cell 2: 환경 설정 ───────────────────────────────────────────
import json, math, asyncio
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import nest_asyncio
nest_asyncio.apply()

try:
    from dotenv import load_dotenv; load_dotenv()
except ImportError: pass

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 환경변수 필요"
hf_token = os.environ.get("HF_TOKEN", "")
if hf_token:
    from huggingface_hub import login; login(hf_token)

assert torch.cuda.is_available()
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
device = torch.device("cuda:0")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Cell 3: 데이터 준비 ─────────────────────────────────────────
def prepare_axis_data(data_json_path, trait_name):
    with open(data_json_path) as f:
        raw = json.load(f)
    instructions = raw.get("instructions", raw.get("instruction", []))
    eval_prompt  = raw.get("eval_prompt",  raw.get("eval prompt", ""))
    # 이중 중괄호 수정
    eval_prompt = eval_prompt.replace("{{question}}", "{question}").replace("{{answer}}", "{answer}")
    questions = raw["questions"]
    assert len(questions) >= 40
    base = {"trait_name": trait_name, "instructions": instructions,
            "eval_prompt": eval_prompt}
    for version, qs in [("extract", questions[:20]), ("eval", questions[20:])]:
        out_dir = Path(f"data_generation/trait_data_{version}")
        out_dir.mkdir(parents=True, exist_ok=True)
        with open(out_dir / f"{trait_name}.json", "w") as f:
            json.dump({**base, "questions": qs}, f, indent=2, ensure_ascii=False)
        print(f"  [{version}] {len(qs)}문항 저장")

extract_json = Path(f"data_generation/trait_data_extract/{TRAIT_NAME}.json")
eval_json    = Path(f"data_generation/trait_data_eval/{TRAIT_NAME}.json")

if not extract_json.exists() or not eval_json.exists():
    prepare_axis_data(DATA_JSON, TRAIT_NAME)

with open(extract_json) as f:
    extract_data = json.load(f)
with open(eval_json) as f:
    eval_data = json.load(f)

EXTRACT_QUESTIONS = extract_data["questions"]
EVAL_QUESTIONS    = eval_data["questions"]
INSTRUCTIONS      = extract_data["instructions"]
EVAL_PROMPT       = extract_data["eval_prompt"].replace(
    "{{question}}", "{question}").replace("{{answer}}", "{answer}")

assert "{question}" in EVAL_PROMPT and "{answer}" in EVAL_PROMPT, "eval_prompt placeholder 오류"
print(f"extract: {len(EXTRACT_QUESTIONS)}, eval: {len(EVAL_QUESTIONS)}, "
      f"instruction pairs: {len(INSTRUCTIONS)}")

In [ ]:
# ── Cell 4: Judge 정의 ──────────────────────────────────────────
COHERENCE_PROMPT = (
    "I am evaluating a different language model. I want to assess the coherence of its answer.\n"
    "I asked the model:\n[QUESTION START]\n{question}\n[QUESTION END]\n"
    "The model answered:\n[ANSWER START]\n{answer}\n[ANSWER END]\n"
    "Rate 0-100 for coherence only (0=gibberish, 50=rambling, 100=perfectly clear).\n"
    "Respond with a number only."
)

class OpenAiJudge:
    """0-100 judge, logprob-weighted (Betley et al. 2025)."""
    def __init__(self, model_name, prompt_template):
        from openai import AsyncOpenAI
        self.client = AsyncOpenAI()
        self.model = model_name
        self.prompt_template = prompt_template

    async def __call__(self, **kwargs):
        messages = [{"role": "user",
                     "content": self.prompt_template.format(**kwargs)}]
        c = await self.client.chat.completions.create(
            model=self.model, messages=messages,
            max_tokens=1, temperature=0,
            logprobs=True, top_logprobs=20, seed=0)
        try:
            lp = c.choices[0].logprobs.content[0].top_logprobs
        except (IndexError, AttributeError):
            return None
        probs = {el.token: float(math.exp(el.logprob)) for el in lp}
        total = s = 0.0
        for k, v in probs.items():
            try:
                ik = int(k)
            except ValueError:
                continue
            if 0 <= ik <= 100:
                s += ik * v; total += v
        return None if total < 0.25 else s / total


async def score_batch(judge, questions, answers, sem):
    async def _one(q, a):
        async with sem:
            return await judge(question=q, answer=a)
    return await asyncio.gather(*[_one(q, a) for q, a in zip(questions, answers)])

print("Judge 정의 완료")

In [ ]:
# ── Cell 5: 모델 로드 ───────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

print(f"모델 로드: {cfg['model_id']}")
tokenizer = AutoTokenizer.from_pretrained(
    cfg["model_id"], trust_remote_code=cfg["trust_remote_code"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    cfg["model_id"], torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"}, trust_remote_code=cfg["trust_remote_code"])
model.eval()
print(f"로드 완료: {next(model.parameters()).device}, "
      f"레이어 수: {model.config.num_hidden_layers}")

In [ ]:
# ── 공통 헬퍼 (모든 노트북 동일) ─────────────────────────────────
# 수정사항 반영:
#   (1) add_special_tokens=False 통일 (Llama BOS 중복 방지)
#   (2) generation slicing: input_ids.shape[1] 기준 (left padding 대응)
#   (3) unit_vector: 벡터 norm 통제
#   (4) dual-basis projection: 비직교 2축 좌표 정확 계산

def tokenize_chat_batch(messages_batch):
    """chat 배치 -> (prompt_texts, inputs). add_special_tokens=False 통일."""
    prompt_texts = [
        tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        for messages in messages_batch
    ]
    inputs = tokenizer(
        prompt_texts, return_tensors="pt",
        padding=True, truncation=True,
        add_special_tokens=False,
    ).to(device)
    return prompt_texts, inputs


def tokenize_chat_single(messages):
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(
        prompt_text, return_tensors="pt",
        add_special_tokens=False,
    ).to(device)
    return prompt_text, inputs


def decode_generated_batch(output_ids, inputs):
    """left padding과 무관하게 새로 생성된 토큰만 디코딩."""
    input_width = inputs["input_ids"].shape[1]
    return tokenizer.batch_decode(
        output_ids[:, input_width:], skip_special_tokens=True)


def unit_vector(v, eps=1e-12):
    v = v.float()
    return v / v.norm().clamp_min(eps)


def make_dual_projector(v_econ, v_cult, cond_limit=100.0):
    """비직교 2축의 dual-basis projector. coords = P @ h"""
    u_e = unit_vector(v_econ)
    u_c = unit_vector(v_cult)
    basis = torch.stack([u_e, u_c], dim=1)          # (hidden, 2)
    gram = basis.T @ basis                            # (2, 2)
    cond = float(torch.linalg.cond(gram))
    assert cond < cond_limit, f"ill-conditioned basis: cond={cond:.1f}"
    return torch.linalg.solve(gram, basis.T)          # (2, hidden)


def project_2d(hidden_state, dual_projector):
    coords = dual_projector @ hidden_state.float()
    return float(coords[0]), float(coords[1])


class LastTokenSteerer:
    """마지막 토큰 위치에 alpha * original_hidden_norm * unit_vector 주입."""
    def __init__(self, model, vector, alpha, layer_1based):
        self.model = model
        self.alpha = float(alpha)
        self.layer_idx = layer_1based - 1
        self.vec = unit_vector(vector)
        self._handle = None

    def _hook(self, module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        h = h.clone()

        v = self.vec.to(device=h.device, dtype=h.dtype)

        original_norm = (
            h[:, -1, :]
            .norm(dim=-1, keepdim=True)
            .clamp_min(1e-8)
        )

        h[:, -1, :] += self.alpha * original_norm * v

        return (h,) + out[1:] if isinstance(out, tuple) else h

    def __enter__(self):
        self._handle = self.model.model.layers[
            self.layer_idx
        ].register_forward_hook(self._hook)
        return self

    def __exit__(self, *_):
        if self._handle is not None:
            self._handle.remove()
            self._handle = None


print("공통 헬퍼 정의 완료 (add_special_tokens=False / input_width slicing / "
      "unit vector / dual-basis / last-token steering)")

In [ ]:
# ── Cell 7: Phase A — pos/neg 응답 생성 + Judge scoring FAST ─────
# 계산은 기존과 동일:
# - 동일 generation
# - 동일 judge model
# - 동일 EVAL_PROMPT
# - 동일 COHERENCE_PROMPT
# - 각 response를 개별적으로 judge
#
# 변경:
# - trait judge와 coherence judge를 동시에 실행
# - 각각 독립 semaphore 사용
# - generation 직후 raw response 임시 저장

@torch.inference_mode()
def generate_batch(message_pairs, max_new_tokens, seed):
    """(system, question) 쌍 -> (prompt_texts, answers)."""
    messages_batch = []
    for system, question in message_pairs:
        m = []
        if system:
            m.append({"role": "system", "content": system})
        m.append({"role": "user", "content": question})
        messages_batch.append(m)

    prompt_texts, inputs = tokenize_chat_batch(messages_batch)

    torch.manual_seed(seed)
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        out = model.generate(
            **inputs,
            generation_config=GenerationConfig(
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=1.0,
                top_p=1.0,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        )

    answers = decode_generated_batch(out, inputs)
    return prompt_texts, answers


def run_generation_and_scoring(persona_type):
    out_dir = Path("output_raw") / cfg["short_name"]
    out_dir.mkdir(parents=True, exist_ok=True)

    csv_path = out_dir / f"{TRAIT_NAME}_{persona_type}.csv"
    raw_csv_path = out_dir / f"{TRAIT_NAME}_{persona_type}_generated_only.csv"

    # 이미 최종 파일 있으면 재사용
    if SKIP_GENERATION and csv_path.exists():
        print(f"  [{persona_type}] 재사용: {csv_path}")
        return csv_path

    # Judges
    judge_trait = OpenAiJudge(JUDGE_MODEL, EVAL_PROMPT)
    judge_coh = OpenAiJudge(JUDGE_MODEL, COHERENCE_PROMPT)

    # 각각 독립적으로 concurrency 사용
    sem_trait = asyncio.Semaphore(MAX_CONCURRENT_JUDGES)
    sem_coh = asyncio.Semaphore(MAX_CONCURRENT_JUDGES)

    # Jobs
    jobs = []
    for q_idx, question in enumerate(EXTRACT_QUESTIONS):
        for k, instr_pair in enumerate(INSTRUCTIONS):
            for r in range(N_PER_Q):
                jobs.append({
                    "system": instr_pair[persona_type],
                    "question": question,
                    "q_idx": q_idx,
                    "instr_idx": k,
                    "rollout": r,
                })

    # Generation 재사용 가능
    if raw_csv_path.exists():
        print(f"  [{persona_type}] 기존 generation 재사용: {raw_csv_path}")
        raw_df = pd.read_csv(raw_csv_path)
        rows = raw_df.to_dict("records")

    else:
        print(f"  [{persona_type}] {len(jobs)}개 생성 (do_sample=True, seeded)...")
        rows = []

        for i in tqdm(
            range(0, len(jobs), BATCH_SIZE),
            desc=f"  gen({persona_type})"
        ):
            batch = jobs[i:i + BATCH_SIZE]
            pairs = [(j["system"], j["question"]) for j in batch]

            # 기존 seed 방식 그대로
            seed = 1000 + (0 if persona_type == "pos" else 500) + i

            prompts, answers = generate_batch(
                pairs,
                MAX_NEW_TOKENS,
                seed,
            )

            for j, job in enumerate(batch):
                rows.append({
                    "question": job["question"],
                    "prompt": prompts[j],
                    "answer": answers[j],
                    "q_idx": job["q_idx"],
                    "instr_idx": job["instr_idx"],
                    "rollout": job["rollout"],
                })

        # generation 완료 즉시 저장
        pd.DataFrame(rows).to_csv(raw_csv_path, index=False)
        print(f"  [{persona_type}] generation 저장 완료: {raw_csv_path}")

    # Scoring
    print(f"  [{persona_type}] scoring {len(rows)}개...")

    questions = [r["question"] for r in rows]
    answers = [r["answer"] for r in rows]

    async def run_both_judges():
        trait_task = score_batch(
            judge_trait,
            questions,
            answers,
            sem_trait,
        )

        coherence_task = score_batch(
            judge_coh,
            questions,
            answers,
            sem_coh,
        )

        trait_scores, coh_scores = await asyncio.gather(
            trait_task,
            coherence_task,
        )

        return trait_scores, coh_scores

    trait_scores, coh_scores = (
        asyncio.get_event_loop()
        .run_until_complete(run_both_judges())
    )

    # Save
    df = pd.DataFrame(rows)
    df[TRAIT_NAME] = trait_scores
    df["coherence"] = coh_scores
    df.to_csv(csv_path, index=False)

    valid = df[TRAIT_NAME].dropna()

    print(
        f"  [{persona_type}] "
        f"{TRAIT_NAME}: "
        f"{valid.mean():.1f} ± {valid.std():.1f}"
    )

    return csv_path


pos_csv = run_generation_and_scoring("pos")
neg_csv = run_generation_and_scoring("neg")

In [ ]:
# ── Cell 8: Phase B — Hidden state 추출 → Persona vector ────────
# 수정: pos/neg key merge / FP32 평균 / add_special_tokens=False
if SKIP_VECTOR:
    print("SKIP_VECTOR=True")
else:
    @torch.inference_mode()
    def extract_hidden_states(prompts, responses):
        n_layers = model.config.num_hidden_layers + 1
        pa = [[] for _ in range(n_layers)]
        ra = [[] for _ in range(n_layers)]
        pl = [[] for _ in range(n_layers)]

        for prompt, response in tqdm(zip(prompts, responses),
                                     total=len(prompts), desc="  hidden"):
            full_text = prompt + response
            inputs = tokenizer(full_text, return_tensors="pt",
                               add_special_tokens=False).to(device)
            prompt_len = len(tokenizer.encode(prompt, add_special_tokens=False))

            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                outputs = model(**inputs, output_hidden_states=True)

            for layer in range(n_layers):
                hs = outputs.hidden_states[layer][0].float()   # FP32 변환
                p_tok = hs[:prompt_len]
                r_tok = hs[prompt_len:]
                if r_tok.shape[0] == 0:
                    r_tok = p_tok[-1:]   # empty response guard
                pa[layer].append(p_tok.mean(dim=0, keepdim=True).cpu())
                ra[layer].append(r_tok.mean(dim=0, keepdim=True).cpu())
                pl[layer].append(p_tok[-1].unsqueeze(0).cpu())
            del outputs
            if len(pa[0]) % 50 == 0:
                torch.cuda.empty_cache()

        return ([torch.cat(x, 0) for x in pa],
                [torch.cat(x, 0) for x in pl],
                [torch.cat(x, 0) for x in ra])


    # ── pos/neg를 key로 정확히 pairing ──
    pos_df = pd.read_csv(pos_csv)
    neg_df = pd.read_csv(neg_csv)
    PAIR_KEYS = ["q_idx", "instr_idx", "rollout"]

    paired = pos_df.merge(neg_df, on=PAIR_KEYS, how="inner",
                          suffixes=("_pos", "_neg"), validate="one_to_one")
    assert (paired["question_pos"] == paired["question_neg"]).all()
    print(f"paired: {len(paired)} (pos={len(pos_df)}, neg={len(neg_df)})")

    mask = (
        (paired[f"{TRAIT_NAME}_pos"] >= THRESHOLD) &
        (paired[f"{TRAIT_NAME}_neg"] < (100 - THRESHOLD)) &
        (paired["coherence_pos"] >= 50) &
        (paired["coherence_neg"] >= 50)
    )
    selected = paired.loc[mask]
    print(f"필터 통과: {len(selected)}/{len(paired)}")
    assert len(selected) > 0

    print("Positive hidden states...")
    pos_pa, pos_pl, pos_ra = extract_hidden_states(
        selected["prompt_pos"].tolist(), selected["answer_pos"].tolist())
    print("Negative hidden states...")
    neg_pa, neg_pl, neg_ra = extract_hidden_states(
        selected["prompt_neg"].tolist(), selected["answer_neg"].tolist())

    n_layers = model.config.num_hidden_layers + 1
    response_avg_diff = torch.stack(
        [pos_ra[l].float().mean(0) - neg_ra[l].float().mean(0) for l in range(n_layers)])
    prompt_avg_diff = torch.stack(
        [pos_pa[l].float().mean(0) - neg_pa[l].float().mean(0) for l in range(n_layers)])
    prompt_last_diff = torch.stack(
        [pos_pl[l].float().mean(0) - neg_pl[l].float().mean(0) for l in range(n_layers)])

    save_dir = Path("persona_vectors") / cfg["short_name"]
    save_dir.mkdir(parents=True, exist_ok=True)
    torch.save(response_avg_diff, save_dir / f"{TRAIT_NAME}_response_avg_diff.pt")
    torch.save(prompt_avg_diff,   save_dir / f"{TRAIT_NAME}_prompt_avg_diff.pt")
    torch.save(prompt_last_diff,  save_dir / f"{TRAIT_NAME}_prompt_last_diff.pt")
    print(f"저장: {save_dir}/ shape={prompt_avg_diff.shape}")

In [ ]:
# ── Cell 9 FAST: Phase C — 레이어 탐색 (VECTOR_TYPE 기준) ────────
#
# 네 현재 vector indexing 구조 기준:
#
#   all_vectors[0]  = embedding output
#   all_vectors[1]  = transformer layer 1 output
#   ...
#   all_vectors[32] = transformer layer 32 output
#
# 따라서:
#
#   displayed layer = layer
#   vector           = all_vectors[layer]
#   hook             = model.model.layers[layer - 1]
#
# 즉 vec = all_vectors[layer] 가 맞음.
#
# 속도 개선:
# alpha들을 하나의 GPU batch에서 동시에 generation.
# ================================================================

if SKIP_VECTOR:
    print("SKIP_VECTOR=True")

else:
    ALPHAS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60]

    LAYER_SEARCH_QUESTIONS = EVAL_QUESTIONS[:10]

    # 10이면 alpha 10개를 한 번에 처리
    # questions=10이면 effective batch = 100
    # OOM이면 5 또는 2로 낮추기
    ALPHA_BATCH_SIZE = 10

    # ============================================================
    # LOAD VECTORS
    # ============================================================

    vec_dir = Path("persona_vectors") / cfg["short_name"]

    all_vectors = torch.load(
        vec_dir / f"{TRAIT_NAME}_{VECTOR_TYPE}_diff.pt",
        weights_only=False,
    )

    num_layers = all_vectors.shape[0]

    print(f"레이어 탐색: {VECTOR_TYPE}, shape={all_vectors.shape}")
    print(f"Transformer layers: {model.config.num_hidden_layers}")
    print(f"Stored vector states: {num_layers}")
    print(f"Alpha batch size: {ALPHA_BATCH_SIZE}")

    # ------------------------------------------------------------
    # indexing sanity check
    # ------------------------------------------------------------

    assert num_layers == model.config.num_hidden_layers + 1, (
        "Expected one embedding state + one vector per transformer layer, "
        f"but got vectors={num_layers}, transformer_layers={model.config.num_hidden_layers}"
    )

    print("\nIndexing check:")
    for layer in [1, 2, model.config.num_hidden_layers]:
        print(
            f"  displayed layer {layer}: vector index={layer}, hook index={layer - 1}"
        )

    # ============================================================
    # JUDGE
    # ============================================================

    judge_trait = OpenAiJudge(JUDGE_MODEL, EVAL_PROMPT)
    sem = asyncio.Semaphore(MAX_CONCURRENT_JUDGES)

    # ============================================================
    # BATCH-SPECIFIC ALPHA STEERER
    #
    # 기존 LastTokenSteerer는 alpha 하나만 받지만,
    # 이 버전은 batch row별로 서로 다른 alpha를 사용.
    #
    # 같은 vector 방향 + 서로 다른 steering strength.
    # ============================================================

    class BatchedAlphaLastTokenSteerer:
        def __init__(self, model, vector, alphas, layer_1based):
            self.model = model
            self.layer_idx = layer_1based - 1
            self.vec = unit_vector(vector)
            self.alphas = torch.as_tensor(alphas, dtype=torch.float32)
            self._handle = None

        def _hook(self, module, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            h = h.clone()

            batch_size = h.shape[0]

            if batch_size != len(self.alphas):
                raise RuntimeError(
                    f"Hook batch mismatch: hidden batch={batch_size}, "
                    f"alpha batch={len(self.alphas)}"
                )

            v = self.vec.to(device=h.device, dtype=h.dtype)
            alpha = self.alphas.to(device=h.device, dtype=h.dtype).view(batch_size, 1)

            h_norm = h[:, -1, :].norm(dim=-1, keepdim=True).clamp_min(1e-8)
            h[:, -1, :] += alpha * h_norm * v.view(1, -1)

            if isinstance(out, tuple):
                return (h,) + out[1:]
            return h

        def __enter__(self):
            self._handle = self.model.model.layers[
                self.layer_idx
            ].register_forward_hook(self._hook)
            return self

        def __exit__(self, exc_type, exc_value, traceback):
            if self._handle is not None:
                self._handle.remove()
                self._handle = None

    # ============================================================
    # GENERATE MULTIPLE ALPHAS IN ONE GPU BATCH
    # ============================================================

    @torch.inference_mode()
    def generate_alpha_batch(questions, layer_1based, vector, alpha_chunk):
        expanded_questions = []
        expanded_alphas = []

        # order:
        # alpha1: q1 q2 ... qN
        # alpha2: q1 q2 ... qN
        # ...
        for alpha in alpha_chunk:
            expanded_questions.extend(questions)
            expanded_alphas.extend([float(alpha)] * len(questions))

        msgs = [[{"role": "user", "content": q}] for q in expanded_questions]

        _, inputs = tokenize_chat_batch(msgs)

        with BatchedAlphaLastTokenSteerer(
            model=model,
            vector=vector,
            alphas=expanded_alphas,
            layer_1based=layer_1based,
        ):
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                out = model.generate(
                    **inputs,
                    generation_config=GenerationConfig(
                        max_new_tokens=256,
                        do_sample=False,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                    ),
                )

        answers = decode_generated_batch(out, inputs)

        # --------------------------------------------------------
        # split answers back by alpha
        # --------------------------------------------------------

        n_q = len(questions)
        results = {}
        idx = 0

        for alpha in alpha_chunk:
            results[float(alpha)] = answers[idx : idx + n_q]
            idx += n_q

        del out
        del inputs
        torch.cuda.empty_cache()

        return results

    # ============================================================
    # LAYER SWEEP
    #
    # IMPORTANT:
    # num_layers = transformer layers + 1
    # range(1, num_layers) => 1 ... 32
    # vec = all_vectors[layer]
    #
    # hook: LastTokenSteerer convention layer_1based - 1
    # ============================================================

    layer_results = {}

    for layer in tqdm(range(1, num_layers), desc="layer sweep"):
        # --------------------------------------------------------
        # CORRECT indexing for this notebook
        # --------------------------------------------------------
        vec = all_vectors[layer]
        generated_by_alpha = {}

        # --------------------------------------------------------
        # alpha batching
        # --------------------------------------------------------
        for alpha_start in range(0, len(ALPHAS), ALPHA_BATCH_SIZE):
            alpha_chunk = ALPHAS[alpha_start : alpha_start + ALPHA_BATCH_SIZE]

            chunk_results = generate_alpha_batch(
                questions=LAYER_SEARCH_QUESTIONS,
                layer_1based=layer,
                vector=vec,
                alpha_chunk=alpha_chunk,
            )

            generated_by_alpha.update(chunk_results)

        # ========================================================
        # JUDGE
        # ========================================================

        qs_all = []
        as_all = []

        for alpha in ALPHAS:
            answers = generated_by_alpha[float(alpha)]
            qs_all.extend(LAYER_SEARCH_QUESTIONS)
            as_all.extend(answers)

        scores = asyncio.get_event_loop().run_until_complete(
            score_batch(judge_trait, qs_all, as_all, sem)
        )

        # ========================================================
        # alpha mean scores
        # ========================================================

        alpha_scores = {}
        idx = 0
        n_q = len(LAYER_SEARCH_QUESTIONS)

        for alpha in ALPHAS:
            chunk = scores[idx : idx + n_q]
            valid = [s for s in chunk if s is not None]

            alpha_scores[float(alpha)] = float(np.mean(valid)) if valid else 0.0
            idx += n_q

        layer_results[layer] = alpha_scores

    # ============================================================
    # BEST LAYER
    #
    # 기존 selection rule 그대로:
    # layer별 alpha 최고값 그 중 전체 최고 layer 선택
    # ============================================================

    best_per_layer = {
        layer: max(alpha_scores.values())
        for layer, alpha_scores in layer_results.items()
    }

    BEST_LAYER = max(best_per_layer, key=best_per_layer.get)
    BEST_ALPHA = max(
        layer_results[BEST_LAYER], key=layer_results[BEST_LAYER].get
    )

    print("\n" + "=" * 80)
    print(f"최적 레이어: {BEST_LAYER}")
    print(f"최적 alpha: {BEST_ALPHA}")
    print(f"최고 score: {best_per_layer[BEST_LAYER]:.2f}")
    print("=" * 80)

    # ============================================================
    # SAVE
    # ============================================================

    torch.save(
        {
            "model": cfg["short_name"],
            "model_id": cfg["model_id"],
            "trait_name": TRAIT_NAME,
            "vector_type": VECTOR_TYPE,
            "best_layer": BEST_LAYER,
            "best_alpha": BEST_ALPHA,
            "layer_scores": best_per_layer,
            "all_results": layer_results,
            "selection_questions": LAYER_SEARCH_QUESTIONS,
            "alphas": ALPHAS,
            "vector_indexing": (
                "all_vectors[0]=embedding; "
                "all_vectors[layer]=transformer layer output"
            ),
            "steering": "last_token_unit_vector_batched_alpha",
            "alpha_batch_size": ALPHA_BATCH_SIZE,
        },
        vec_dir / f"{TRAIT_NAME}_{VECTOR_TYPE}_best_layer_info.pt",
    )

    # ============================================================
    # FIGURE
    # ============================================================

    fig, ax = plt.subplots(figsize=(11, 5))

    colors = plt.cm.viridis(np.linspace(0, 1, len(ALPHAS)))
    layers = sorted(layer_results.keys())

    for i, alpha in enumerate(ALPHAS):
        ax.plot(
            layers,
            [layer_results[layer][float(alpha)] for layer in layers],
            marker="o",
            markersize=3,
            color=colors[i],
            linewidth=1.2,
            alpha=0.8,
        )

    ax.axvline(
        BEST_LAYER,
        color="red",
        linestyle="--",
        linewidth=2,
        label=f"Best layer = {BEST_LAYER}",
    )

    sm = plt.cm.ScalarMappable(
        cmap="viridis",
        norm=plt.Normalize(vmin=min(ALPHAS), vmax=max(ALPHAS)),
    )

    plt.colorbar(sm, ax=ax, label="Steering α (unit vector)")

    ax.set_xlabel("Layer")
    ax.set_ylabel(f"{TRAIT_LABEL} Score")
    ax.set_title(
        f"Layer-wise Steering — {TRAIT_LABEL} ({cfg['short_name']}, {VECTOR_TYPE})"
    )
    ax.legend()
    ax.set_ylim(0, 100)
    ax.grid(alpha=0.3)

    plt.tight_layout()

    plt.savefig(
        vec_dir / f"layer_steering_{TRAIT_NAME}_{VECTOR_TYPE}.png",
        dpi=150,
        bbox_inches="tight",
    )

    plt.show()

In [ ]:
# ── Cell 10: 요약 ───────────────────────────────────────────────
vec_dir = Path("persona_vectors") / cfg["short_name"]
info = torch.load(vec_dir / f"{TRAIT_NAME}_{VECTOR_TYPE}_best_layer_info.pt",
                  weights_only=False)
assert info["model"] == cfg["short_name"]
assert info["vector_type"] == VECTOR_TYPE

print("=" * 55)
print(f"완료: {cfg['short_name']} / {TRAIT_LABEL} / {VECTOR_TYPE}")
print(f"  best_layer: {info['best_layer']}")
print("=" * 55)
print("다음:")
if AXIS == "economic":
    print("  → AXIS='social' 바꾸고 커널 재시작 후 재실행")
else:
    print("  → 두 축 완료. 02_validate_vectors.ipynb")